# GLM-OCR Test Notebook

Bu notebook GLM-OCR modelini Google Colab'da (T4 GPU) test eder.

**Pipeline:** PDF/Görüntü → GLM-OCR (GPU) → Metin → Anonimizasyon → Temiz metin

**Gereksinimler:** Colab T4 GPU (ücretsiz tier yeterli)

---

## 1. Kurulum

In [ ]:
# GPU kontrolü
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# GLM-OCR ve bağımlılıkları kur
!pip install -q "Pillow>=10.0.0,<11.0.0"
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q accelerate PyMuPDF pytesseract
!apt-get install -y -q tesseract-ocr tesseract-ocr-tur > /dev/null 2>&1

# Doğrulama
import torch
print(f"CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print("Kurulum tamamlandı!")

## 2. Dosya Yükleme

PDF veya görüntü dosyalarınızı yükleyin.

In [ ]:
import os
from google.colab import files

# Klasör oluştur
os.makedirs("input_files", exist_ok=True)
os.makedirs("output", exist_ok=True)

# Dosya yükle
print("PDF veya görüntü dosyalarınızı yükleyin:")
uploaded = files.upload()

for filename in uploaded:
    with open(f"input_files/{filename}", "wb") as f:
        f.write(uploaded[filename])
    print(f"  ✓ {filename} ({len(uploaded[filename])//1024} KB)")

print(f"\nToplam {len(uploaded)} dosya yüklendi.")

## 3. GLM-OCR ile Metin Çıkarma (GPU)

In [ ]:
import fitz  # PyMuPDF
from PIL import Image
import time
from pathlib import Path
import torch
import base64
import io


def pdf_to_images(pdf_path, dpi=150):
    """PDF sayfalarını görüntülere çevir. DPI=150 VRAM tasarrufu için."""
    doc = fitz.open(pdf_path)
    images = []
    for page in doc:
        pix = page.get_pixmap(dpi=dpi)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        images.append(img)
    doc.close()
    return images


def load_image(file_path):
    """Dosyayı yükle — PDF ise sayfalara böl, görüntüyse direkt aç."""
    ext = Path(file_path).suffix.lower()
    if ext == ".pdf":
        return pdf_to_images(file_path)
    else:
        return [Image.open(file_path).convert("RGB")]


print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print("\nYüklenen dosyalar:")
for f in sorted(os.listdir("input_files")):
    print(f"  - {f}")

In [ ]:
# ── GLM-OCR modelini yükle (doğrudan transformers ile) ──

from transformers import AutoProcessor, AutoModelForImageTextToText
import torch, time

MODEL_PATH = "zai-org/GLM-OCR"

print("GLM-OCR modeli yükleniyor (ilk seferde ~2 dk indirme)...")
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
)
print(f"✓ Model yüklendi! ({time.time()-t0:.1f}s)")
print(f"  Device: {model.device}")
print(f"  Dtype: {model.dtype}")

In [ ]:
# ── GLM-OCR ile OCR ──

def glm_ocr_single(image: Image.Image) -> str:
    """Tek bir görüntüyü GLM-OCR ile OCR yap."""
    import tempfile, gc

    # T4 VRAM (15GB) için görüntüyü küçült
    # Max 1024px uzun kenar — VRAM kullanımını ~4GB'a düşürür
    MAX_DIM = 1024
    if max(image.size) > MAX_DIM:
        ratio = MAX_DIM / max(image.size)
        new_size = (int(image.width * ratio), int(image.height * ratio))
        image = image.resize(new_size, Image.LANCZOS)
        print(f"(resized to {new_size})", end=" ", flush=True)

    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    image.save(tmp.name)

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "url": tmp.name},
            {"type": "text", "text": "Text Recognition:"}
        ]
    }]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)
    inputs.pop("token_type_ids", None)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=8192)

    output = processor.decode(
        generated_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    os.unlink(tmp.name)

    # VRAM temizle (sonraki sayfa için)
    del inputs, generated_ids
    gc.collect()
    torch.cuda.empty_cache()

    return output


def glm_ocr_file(file_path: str) -> str:
    """Dosyayı (PDF veya görüntü) GLM-OCR ile OCR yap."""
    images = load_image(file_path)
    all_text = []
    for i, img in enumerate(images):
        print(f"  Sayfa {i+1}/{len(images)}...", end=" ", flush=True)
        t0 = time.time()
        try:
            text = glm_ocr_single(img)
            print(f"{time.time()-t0:.1f}s ({len(text)} karakter)")
        except Exception as e:
            text = f"[HATA: {e}]"
            print(f"HATA: {e}")
        all_text.append(text)
    return "\n\n".join(all_text)


# Tüm dosyaları işle
glm_results = {}
for filename in sorted(os.listdir("input_files")):
    filepath = f"input_files/{filename}"
    print(f"\n{'='*50}")
    print(f"GLM-OCR: {filename}")
    print("="*50)
    t0 = time.time()
    text = glm_ocr_file(filepath)
    elapsed = time.time() - t0
    glm_results[filename] = text

    stem = Path(filename).stem
    with open(f"output/{stem}_glmocr.txt", "w", encoding="utf-8") as f:
        f.write(text)

    print(f"\n--- Çıktı ({elapsed:.1f}s, {len(text)} karakter) ---")
    print(text[:800])
    print("...")

## 4. Tesseract OCR ile Karşılaştırma

In [ ]:
import pytesseract


def tesseract_ocr_file(file_path: str) -> str:
    """Dosyayı Tesseract ile OCR yap."""
    images = load_image(file_path)
    all_text = []
    for i, img in enumerate(images):
        text = pytesseract.image_to_string(img, lang="tur", config="--psm 6")
        all_text.append(text)
    return "\n\n".join(all_text)


# Tüm dosyaları Tesseract ile işle
tess_results = {}
for filename in sorted(os.listdir("input_files")):
    filepath = f"input_files/{filename}"
    print(f"Tesseract: {filename}...", end=" ")
    t0 = time.time()
    text = tesseract_ocr_file(filepath)
    elapsed = time.time() - t0
    tess_results[filename] = text

    stem = Path(filename).stem
    with open(f"output/{stem}_tesseract.txt", "w", encoding="utf-8") as f:
        f.write(text)

    print(f"{elapsed:.1f}s, {len(text)} karakter")

## 5. Karşılaştırma Tablosu

In [ ]:
from IPython.display import HTML

# Karşılaştırma tablosu
rows = []
for filename in sorted(glm_results.keys()):
    glm_text = glm_results.get(filename, "")
    tess_text = tess_results.get(filename, "")

    # Basit kalite metrikleri
    glm_words = len(glm_text.split())
    tess_words = len(tess_text.split())

    # Türkçe karakter oranı (OCR kalite göstergesi)
    tr_chars = set("çğıöşüÇĞİÖŞÜ")
    glm_tr = sum(1 for c in glm_text if c in tr_chars)
    tess_tr = sum(1 for c in tess_text if c in tr_chars)

    rows.append({
        "Dosya": filename[:30],
        "GLM-OCR (kelime)": glm_words,
        "Tesseract (kelime)": tess_words,
        "GLM Türkçe karakter": glm_tr,
        "Tess Türkçe karakter": tess_tr,
    })

# Tablo
html = "<table border='1' style='border-collapse:collapse; font-size:14px;'>"
html += "<tr>" + "".join(f"<th style='padding:8px; background:#f0f0f0;'>{k}</th>" for k in rows[0].keys()) + "</tr>"
for row in rows:
    html += "<tr>" + "".join(f"<td style='padding:8px;'>{v}</td>" for v in row.values()) + "</tr>"
html += "</table>"
display(HTML(html))

In [ ]:
# Yan yana metin karşılaştırması (ilk dosya)
from IPython.display import HTML

first_file = sorted(glm_results.keys())[0]
glm_text = glm_results[first_file]
tess_text = tess_results[first_file]

html = f"""
<h3>{first_file}</h3>
<div style='display:flex; gap:20px;'>
  <div style='flex:1; border:1px solid #ccc; padding:10px; max-height:600px; overflow:auto;'>
    <h4 style='color:green;'>GLM-OCR</h4>
    <pre style='white-space:pre-wrap; font-size:12px;'>{glm_text[:3000]}</pre>
  </div>
  <div style='flex:1; border:1px solid #ccc; padding:10px; max-height:600px; overflow:auto;'>
    <h4 style='color:blue;'>Tesseract</h4>
    <pre style='white-space:pre-wrap; font-size:12px;'>{tess_text[:3000]}</pre>
  </div>
</div>
"""
display(HTML(html))

## 6. Çıktıları İndir

In [ ]:
# Tüm çıktıları zip yapıp indir
import shutil
shutil.make_archive("ocr_results", "zip", "output")
files.download("ocr_results.zip")
print("\n✓ Tüm OCR sonuçları indirildi!")

## 7. Temizlik

In [ ]:
# GPU belleğini temizle
import gc
del model
gc.collect()
torch.cuda.empty_cache()
print("GPU belleği temizlendi.")

---
*Not: Bu notebook doğrudan transformers ile inference yapar, vLLM sunucusu gerekmez.*